In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import random

# CONFIG
base_url = "https://remax.com.eg"
SAVE_EVERY = 100

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
    "Mozilla/5.0 (X11; Linux x86_64)"
]

# DATA
data = []
visited_links = set()

# HELPERS
def get_headers():
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    }

def clean_text(text):
    return re.sub(r"\s+", " ", text).strip() if text else None

def extract_number(text):
    if text:
        match = re.search(r"\d+", text.replace(",", ""))
        return match.group() if match else None
    return None

def split_location(full_address):
    if not full_address:
        return None, None, None

    full_address = full_address.replace("Egypt", "")
    parts = [p.strip() for p in re.split(r",|-|/|\|", full_address) if p.strip()]

    city = parts[-1] if len(parts) >= 1 else None
    loc1 = parts[0] if len(parts) >= 2 else None
    loc2 = parts[1] if len(parts) >= 3 else None

    return city, loc1, loc2


# REQUEST
def get_soup(url, retries=3):
    for _ in range(retries):
        try:
            res = requests.get(url, headers=get_headers(), timeout=15)

            if res.status_code == 200:
                return BeautifulSoup(res.content, "lxml")

        except Exception as e:
            print("Retry error:", e)

        time.sleep(random.uniform(1, 3))

    return None


# SCRAPE PROPERTY
def scrape_property(link):
    soup = get_soup(link)
    if not soup:
        return None

    try:
        # Price
        price_tag = soup.select_one(".price-flex h4")
        price_value = clean_text(price_tag.text) if price_tag else None

        # Location
        loc_div = soup.select_one(".global_color.loc")
        full_address = clean_text(loc_div.get_text(" ")) if loc_div else None

        city, loc1, loc2 = split_location(full_address)

        # Description
        desc = soup.select_one("pre, .description, .property-description")
        desc_text = clean_text(desc.text) if desc else ""

        # Rent / Sale
        rent_sale_val = None
        if "rent" in link.lower():
            rent_sale_val = "Rent"
        elif "sale" in link.lower():
            rent_sale_val = "Sale"
        elif desc_text:
            if "rent" in desc_text.lower():
                rent_sale_val = "Rent"
            elif "sale" in desc_text.lower():
                rent_sale_val = "Sale"

        # Features
        beds = baths = area_val = None

        for f in soup.select(".feature"):
            text = f.get_text().lower()

            if "bed" in text:
                beds = extract_number(text)
            elif "bath" in text:
                baths = extract_number(text)

            img = f.find("img")
            if img and "frame" in img.get("src", ""):
                p = f.find("p")
                if p:
                    area_val = extract_number(p.text)

        # Price per meter
        ppm = None
        try:
            if price_value and area_val:
                price_num = float(re.sub(r"[^\d]", "", price_value))
                area_num = float(area_val)

                if area_num > 0:
                    ppm = round(price_num / area_num, 2)
        except:
            pass

        return {
            "Link": link,
            "Bedrooms": beds,
            "Bathrooms": baths,
            "Area": area_val,
            "Price": price_value,
            "Price_per_meter": ppm,
            "Address": full_address,
            "City": city,
            "Location1": loc1,
            "Location2": loc2 if loc2 else "None",
            "Rent_sale": rent_sale_val
        }

    except Exception as e:
        print("Parse error:", link, e)
        return None


# MAIN LOOP
page = 1

while True:
    print(f"\nPage {page}")

    soup = get_soup(f"{base_url}/listings?page={page}")

    if not soup:
        print("STOP - no page")
        break

    links = []
    for a in soup.select("a[href*='/property/']"):
        href = a.get("href")
        if href:
            link = href if href.startswith("http") else base_url + href
            links.append(link)

    links = list(set(links))
    new_links = [l for l in links if l not in visited_links]

    if not new_links:
        print("DONE")
        break

    print(f"Found {len(new_links)} properties")



    # SEQUENTIAL SCRAPING
    for link in new_links:
        result = scrape_property(link)

        if result:
            data.append(result)
            visited_links.add(result["Link"])

            print("Scraped:", result["Link"])

            # Auto-save
            if len(data) % SAVE_EVERY == 0:
                pd.DataFrame(data).to_csv(
                    "remax_partial.csv",
                    index=False,
                    encoding="utf-8-sig"
                )
                print(f"Saved {len(data)} rows")

        time.sleep(random.uniform(0.5, 1.5))

    page += 1
    time.sleep(random.uniform(1, 2))

# FINAL SAVE
df = pd.DataFrame(data)
df.fillna("None", inplace=True)
df.to_csv("Remax Egypt.csv", index=False, encoding="utf-8-sig")

print("\n DONE SCRAPING")


Page 1
Found 12 properties
Scraped: https://remax.com.eg/property/apartment-for-sale-in-9th-district-open-view-9104
Scraped: https://remax.com.eg/property/rtm-apartment-with-best-price-in-village-west-8198
Scraped: https://remax.com.eg/property/twinhouse-for-sale-in-romance-ain-sokhna-8740
Scraped: https://remax.com.eg/property/i-villa-roof-corner-for-sale-mountain-view-icity-8932
Scraped: https://remax.com.eg/property/apartment-for-rent-in-o-west-high-end-finishing-9101
Scraped: https://remax.com.eg/property/furnished-penthouse-for-rent-in-tara-compound-9100
Scraped: https://remax.com.eg/property/clinic-with-installments-prime-location-in-zayed-9099
Scraped: https://remax.com.eg/property/apartment-for-rent-in-janna-zayed-4-prime-location-9102
Scraped: https://remax.com.eg/property/prime-office-for-sale-in-2o5-sheikh-zayed-8541
Scraped: https://remax.com.eg/property/apartment-with-garden-kitchen-and-acs-in-el-joman-8755
Scraped: https://remax.com.eg/property/townhouse-for-sale-in-sole

In [ ]:
# To check the full number of pages on the website

headers = {"User-Agent": "Mozilla/5.0"}
base_url = "https://remax.com.eg"

page = 1

while True:
    url = f"{base_url}/listings?page={page}"
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.content, "lxml")

    properties = soup.find_all("a", href=True)


    if not any("/property/" in a["href"] for a in properties):
        break

    print(f"Page {page} found")
    page += 1

print("Total Pages =", page - 1)

Page 1 found
Page 2 found
Page 3 found
Page 4 found
Page 5 found
Page 6 found
Page 7 found
Page 8 found
Page 9 found
Page 10 found
Page 11 found
Page 12 found
Page 13 found
Page 14 found
Page 15 found
Page 16 found
Page 17 found
Page 18 found
Page 19 found
Page 20 found
Page 21 found
Page 22 found
Page 23 found
Page 24 found
Page 25 found
Page 26 found
Page 27 found
Page 28 found
Page 29 found
Page 30 found
Page 31 found
Page 32 found
Page 33 found
Page 34 found
Page 35 found
Page 36 found
Page 37 found
Page 38 found
Page 39 found
Page 40 found
Page 41 found
Page 42 found
Page 43 found
Page 44 found
Page 45 found
Page 46 found
Page 47 found
Page 48 found
Page 49 found
Page 50 found
Page 51 found
Page 52 found
Page 53 found
Page 54 found
Page 55 found
Page 56 found
Page 57 found
Page 58 found
Page 59 found
Page 60 found
Page 61 found
Page 62 found
Page 63 found
Page 64 found
Page 65 found
Page 66 found
Page 67 found
Page 68 found
Page 69 found
Page 70 found
Page 71 found
Page 72 found
P

In [8]:
# To check the full wanted features of pages on the website
headers = {"User-Agent": "Mozilla/5.0"}
base_url = "https://remax.com.eg"

page = 1
all_links = set()

while True:
    print(f"Checking Page {page}...")

    url = f"{base_url}/listings?page={page}"
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.content, "lxml")

    links = [
        base_url + a["href"]
        for a in soup.find_all("a", href=True)
        if "/property/" in a["href"]
    ]

    links = set(links)

    new_links = links - all_links
    if not new_links:
        break

    all_links.update(new_links)
    print(f"Found {len(new_links)} new links")

    page += 1


print("\n Total Properties (Units) in Website:")
print(len(all_links))

Checking Page 1...
Found 12 new links
Checking Page 2...
Found 12 new links
Checking Page 3...
Found 12 new links
Checking Page 4...
Found 12 new links
Checking Page 5...
Found 12 new links
Checking Page 6...
Found 12 new links
Checking Page 7...
Found 12 new links
Checking Page 8...
Found 12 new links
Checking Page 9...
Found 12 new links
Checking Page 10...
Found 12 new links
Checking Page 11...
Found 12 new links
Checking Page 12...
Found 12 new links
Checking Page 13...
Found 12 new links
Checking Page 14...
Found 12 new links
Checking Page 15...
Found 12 new links
Checking Page 16...
Found 12 new links
Checking Page 17...
Found 12 new links
Checking Page 18...
Found 12 new links
Checking Page 19...
Found 12 new links
Checking Page 20...
Found 12 new links
Checking Page 21...
Found 12 new links
Checking Page 22...
Found 12 new links
Checking Page 23...
Found 12 new links
Checking Page 24...
Found 12 new links
Checking Page 25...
Found 12 new links
Checking Page 26...
Found 12 new l